# Energy Bills Automation

## Import Libraries and Utilities

In [1]:
#!/usr/bin/env python3
from utils.logger import BaseLogger
from utils.dependencies import *
from utils.data_process import _dtypes_convert
from utils.table_reader import *
from utils.queries import pollaploi_query, parohes_cosmote_query 
from utils.type_mapping import type_mapping
from utils.profiler import PipelineProfiler
from utils.send_mail import *

In [2]:
logger = BaseLogger().logger
prof = PipelineProfiler(logger)

In [1]:
import os
os.getenv("POSIT_PRODUCT")

'WORKBENCH'

In [ ]:
def load_creds(ON_CONNECT: bool = os.getenv("POSIT_PRODUCT") == "CONNECT", logger: logging.Logger = None)-> dict:
    """Custom creds loader from env with environment detection capability"""
    
    if ON_CONNECT:
        logger.info(f"Running on {os.getenv('POSIT_PRODUCT')}")
        conn_info = {
                    'host': os.getenv('VERTICA_HOST'),
                    'port': int(os.getenv('VERTICA_PORT', 5433)),
                    'user': os.getenv('VERTICA_USER'),
                    'password': os.getenv('VERTICA_PASSWORD'),
                    'database': os.getenv('VERTICA_DATABASE'),
                    'tlsmode': os.getenv('VERTICA_TLSMODE', 'disable'),}
    else:
        if load_dotenv(dotenv_path=f"{Path(os.getenv('HOME'))}/dev.env",  override=True):
            conn_info = dict(dotenv_values(f"{Path(os.getenv('HOME'))}/dev.env"))
            logger.info("Loaded credentials from environment file")
        else:
            logger.warning("Failed to get credentials from environment file. Reverting to default [if exist]")
            logger.info("Available .env files:\n"+'\n'.join([f for f in os.listdir(os.getenv('HOME')) if '.env' in f]))
            load_dotenv()
            conn_info = dict(dotenv_values())

    # sanity check
    required = ['host', 'port', 'user', 'password', 'database']
    conn_info = {str(re.sub("VERTICA_", "", k)).lower(): v for k, v in conn_info.items()}

    missing = [str(re.sub(pattern="VERTICA_", repl="", string=key)) for key in required if key not in conn_info.keys()]
    if missing:
        logger.error(f"Missing env variables:\n"+'\n'.join(missing))

        conn_info['tlsmode'] = 'disable'
        logger.info("Reverting to default")
    
    return conn_info

In [ ]:
ON_CONNECT = os.getenv("POSIT_PRODUCT") == "CONNECT"

if ON_CONNECT:
    logger.info(f"Running on {os.getenv('POSIT_PRODUCT')}")
    conn_info = {
                'host': os.getenv('VERTICA_HOST'),
                'port': int(os.getenv('VERTICA_PORT', 5433)),
                'user': os.getenv('VERTICA_USER'),
                'password': os.getenv('VERTICA_PASSWORD'),
                'database': os.getenv('VERTICA_DATABASE'),
                'tlsmode': os.getenv('VERTICA_TLSMODE', 'disable'),
    }
    print(conn_info)
else:
    if load_dotenv(dotenv_path=f"{Path(os.getenv('HOME'))}/dev.env",  override=True):
        conn_info = dict(dotenv_values(f"{Path(os.getenv('HOME'))}/dev.env"))
        logger.info("Loaded credentials from environment file")
    else:
        logger.warning("Failed to get credentials from environment file. Reverting to default [if exist]")
        logger.info("Available .env files:\n"+'\n'.join([f for f in os.listdir(os.getenv('HOME')) if '.env' in f]))
        load_dotenv()
        conn_info = dict(dotenv_values())

# sanity check
required = ['host', 'port', 'user', 'password', 'database']
conn_info = {str(re.sub("VERTICA_", "", k)).lower(): v for k, v in conn_info.items()}

missing = [str(re.sub(pattern="VERTICA_", repl="", string=key)) for key in required if key not in conn_info.keys()]
if missing:
    logger.error(f"Missing env variables:\n"+'\n'.join(missing))

    conn_info['tlsmode'] = 'disable'
    logger.info("Reverting to default")

In [3]:
def load_creds(env: str = "", 
               logger: logging.Logger = None) -> dict:
    # check environment 
    pass

22-04-2026 15:03:44 INFO [2136086929.<module>:3]: Loaded credentials from environment file


In [ ]:
with vertica_python.connect(**conn_info) as con:
    with con.cursor() as cur:
        print(pd.DataFrame(cur.execute("select count(*) from energy_efficiency.plpl2").fetchall(), columns=[c[0] for c in cur.description]))

## Import Data

In [2]:
# logger.info(f"Logging started at {datetime.now()}")

# n_workers = 60
# logger.info(f"Preparing to load tables with deault number of concurrent workers: {n_workers}")

# # Read tables
# logger.info("Reading table: energy_efficiency.pollaploi")
# with prof.stage("fetch_pollaploi"):
#     df = read_vertica_table_with_multiprocessing(conn_info, pollaploi_query, num_workers=n_workers, logger=logger)

# logger.info("Reading table: energy_efficiency.parohes_cosmote")
# with prof.stage("fetch_parohes"):
#     sites = read_vertica_table(conn_info, parohes_cosmote_query)

# # Apply basic data processing
# logger.info("Processing data: filling N/A values, applying type casting, and creating required columns.")